# DEM Calibration using Reference Elevation Data

This notebook calibrates the initial relative-height DEM produced by `01_shape_from_shading.ipynb`.

A reference DEM is required to estimate the linear relationship between the reconstructed heights and reference elevations. The reference data must cover the same terrain region and be spatially aligned with the initial DEM.

The required aligned reference DEM is not included as a ready-to-use file in this repository. Provide a suitable reference raster named `reference_dem.tif` before running this notebook.

The calibration output is saved as a floating-point TIFF file. The calibration results depend on the quality and alignment of the reference data.

In [ ]:
from pathlib import Path
import imageio.v2 as imageio
import numpy as np
import matplotlib.pyplot as plt

reference_dem_path = Path("reference_dem.tif")

if not reference_dem_path.is_file():
    raise FileNotFoundError(
        "reference_dem.tif was not found. "
        "Provide a reference DEM that is spatially aligned "
        "with the initial SFS DEM."
    )

z_true_reference = imageio.imread(reference_dem_path).astype(np.float32)

if z_true_reference.ndim != 2:
    raise ValueError("The reference DEM must be a single-channel 2D raster.")

valid_reference = np.isfinite(z_true_reference)

if not np.any(valid_reference):
    raise ValueError("The reference DEM contains no finite elevation values.")

print("Reference DEM shape:", z_true_reference.shape)
print("Reference DEM valid range:",
      np.min(z_true_reference[valid_reference]),
      "to",
      np.max(z_true_reference[valid_reference]))

## Load the Reconstructed Relative DEM

Load the relative DEM generated by the Shape-from-Shading notebook.

Inspect the dimensions and numerical ranges of the reconstructed
and reference DEMs before proceeding with calibration.

In [ ]:
initial_dem_path = Path("initial_predicted_dem.tiff")

if not initial_dem_path.is_file():
    raise FileNotFoundError(
        "initial_predicted_dem.tiff was not found. "
        "Run the DEM reconstruction section in notebook 01 first."
    )

z = imageio.imread(initial_dem_path).astype(np.float32)

if z.ndim != 2:
    raise ValueError("The initial DEM must be a single-channel 2D raster.")

print("Initial DEM shape:", z.shape)
print("Reference DEM shape:", z_true_reference.shape)

if z.shape != z_true_reference.shape:
    raise ValueError(
        "The initial and reference DEMs have different shapes. "
        "Prepare and spatially align the reference DEM before calibration."
    )

print("Array shapes match. Spatial alignment must still be verified.")

## Fit Linear Elevation Calibration

Fit a linear transformation between the reconstructed relative DEM
and the reference DEM using corresponding valid pixels.

The fitted scale and offset are used to map the relative height values
onto the elevation scale of the reference DEM.

The two arrays must have matching dimensions and spatial alignment
before the regression is performed.

In [ ]:
from sklearn.linear_model import LinearRegression

valid_mask = np.isfinite(z) & np.isfinite(z_true_reference)

if np.count_nonzero(valid_mask) < 2:
    raise ValueError("Not enough valid pixels to fit the calibration.")

z_sfs_valid = z[valid_mask]
z_reference_valid = z_true_reference[valid_mask]

reg = LinearRegression()
reg.fit(z_sfs_valid.reshape(-1, 1), z_reference_valid)

a = float(reg.coef_[0])
b = float(reg.intercept_)

print(f"Calibration relationship: Z_abs = {a:.6g} * Z_sfs + {b:.6g}")

z_final_abs = (a * z + b).astype(np.float32)
z_final_abs[~valid_mask] = np.nan

print("Calibrated DEM shape:", z_final_abs.shape)
print("Valid calibrated pixels:", np.count_nonzero(valid_mask))

## Save and Visualize the Calibrated DEM

Save the calibrated elevation map as a floating-point TIFF file.

Compare the calibrated reconstruction with the reference DEM using
side-by-side elevation maps with a shared colour scale.

In [ ]:
import tifffile
import matplotlib.pyplot as plt

# --- Save the calibrated DEM ---

output_path = "predicted_final_absolute_dem.tiff"

tifffile.imwrite(
    output_path,
    z_final_abs.astype(np.float32)
)

print(f"Calibrated DEM saved to: {output_path}")

# --- Compare the calibrated and reference DEMs ---

# Use common valid pixels to define a shared colour scale
comparison_values = np.concatenate([
    z_final_abs[valid_mask],
    z_true_reference[valid_mask]
])

vmin = np.min(comparison_values)
vmax = np.max(comparison_values)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(
    np.ma.masked_invalid(z_final_abs),
    cmap="terrain",
    vmin=vmin,
    vmax=vmax
)
axes[0].set_title("Predicted Absolute DEM")
axes[0].axis("off")

im1 = axes[1].imshow(
    np.ma.masked_invalid(z_true_reference),
    cmap="terrain",
    vmin=vmin,
    vmax=vmax
)
axes[1].set_title("Reference DEM")
axes[1].axis("off")

fig.colorbar(
    im1,
    ax=axes,
    label="Elevation (reference DEM units)",
    shrink=0.85
)

plt.tight_layout()
plt.savefig(
    "dem_calibration_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()